In [44]:
library(ggplot2)
library(data.table)
library(stringr)
theme_set(theme_bw())

In [57]:
tools = c('singlem_r226', 'sylph_r226')
samples_list = list(
    "Micromonospora yangpuensis" = c("SRR22388335", "SRR9650389"),
    "Liquorilactobacillus uvarum" = c("SRR16352837", "SRR16352839", "SRR17498764"),
    "Pandoraea thiooxydans" = c("SRR22870123", "SRR23961386", "SRR24982124"),
    "Chryseobacterium oleae" = "SRR6201989",
    "Pseudomonas_E fulva_B" = c("SRR5264410", "SRR6869034", "SRR5264435")
)
samples = data.table(purrr::list_rbind(purrr::imap(samples_list, function(x, y) data.frame(sample = x, species = y, temp = 1))))
d1 = samples[data.table(tool = tools, temp = 1), on = "temp", allow.cartesian = TRUE]
samples[, temp := NULL]
d1[, temp := NULL]

In [120]:
readit = function(tool, sample){
    to_read = paste0('output_',tool,'/', tool, '/',sample,'.profile')
    return(fread(to_read))
}
d2 = d1[, readit(tool, sample)[, c("coverage", "taxonomy")], by=list(tool, sample, species)]
d2[, ra := coverage / sum(coverage, na.rm = TRUE) * 100, by = list(tool, sample, species)]
d2[, clean_taxonomy := gsub("; ", ";", gsub("Root; ", "", taxonomy))]

In [ ]:
by_tool = dcast(d2, sample + species + clean_taxonomy ~ tool, value.var = c("ra"), fill = 0)
#by_tool[, avg_rdiff := mean(2 * abs(singlem_r226 - sylph_r226) / (singlem_r226 + sylph_r226)), by = list(sample, species)]
by_tool[, bc := 1 - 2 * sum(pmin(singlem_r226, sylph_r226)) / sum(singlem_r226 + sylph_r226), by = list(sample, species)]
by_tool[str_detect(tolower(clean_taxonomy), tolower(species)),
        list(sample, species,
             singlem = round(singlem_r226, 4),
             sylph = round(sylph_r226, 4),
             #rdiff = round(2 * abs(singlem_r226 - sylph_r226) / (singlem_r226 + sylph_r226), 4),
             #avg_rdiff = round(avg_rdiff, 4),
             bc = round(bc, 4))]

sample,species,singlem,sylph,bc
<chr>,<chr>,<dbl>,<dbl>,<dbl>
SRR16352837,Liquorilactobacillus uvarum,0.1273,0.7147,0.8110
SRR16352839,Liquorilactobacillus uvarum,0.1556,0.5695,0.7623
SRR17498764,Liquorilactobacillus uvarum,0.6837,1.1834,0.1388
SRR22388335,Micromonospora yangpuensis,0.0172,0.0000,0.9855
SRR22870123,Pandoraea thiooxydans,0.0645,0.7117,0.8670
SRR23961386,Pandoraea thiooxydans,2.5314,6.2588,0.7483
SRR24982124,Pandoraea thiooxydans,2.5314,6.2588,0.7483
SRR5264410,Pseudomonas_E fulva_B,0.8728,0.6829,0.4659
SRR5264435,Pseudomonas_E fulva_B,0.6877,0.5234,0.4523


# No chryseobacterium oleae

In [70]:
grep("oleae", d2$taxonomy, value = TRUE)
grep("s__Chryseobacterium", d2$taxonomy, value = TRUE)

character(0)

[1] "Root; d__Bacteria; p__Bacteroidota; c__Bacteroidia; o__Flavobacteriales; f__Weeksellaceae; g__Chryseobacterium; s__Chryseobacterium sp000737715"
 [2] "Root; d__Bacteria; p__Bacteroidota; c__Bacteroidia; o__Flavobacteriales; f__Weeksellaceae; g__Chryseobacterium; s__Chryseobacterium sp900128945"
 [3] "Root; d__Bacteria; p__Bacteroidota; c__Bacteroidia; o__Flavobacteriales; f__Weeksellaceae; g__Chryseobacterium; s__Chryseobacterium sp001424585"
 [4] "Root; d__Bacteria; p__Bacteroidota; c__Bacteroidia; o__Flavobacteriales; f__Weeksellaceae; g__Chryseobacterium; s__Chryseobacterium sp001424585"
 [5] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium indoltheticum"          
 [6] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium sp001507335"            
 [7] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium carnipullorum"          
 [8] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium shigense"               
 [9] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium piscium"                
[10] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium vrystaatense"           
[11] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium aquaticum"              
[12] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium aquaticum"              
[13] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium sp000737715"            
[14] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium camelliae"              
[15] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium formosense"             
[16] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium foetidum"               
[17] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium indoltheticum"          
[18] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium sp001424585"            
[19] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium sp001424105"            
[20] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium profundimaris"          
[21] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium arachidis"              
[22] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium sp003182335"            
[23] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium luteum"                 
[24] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium sp019203745"            
[25] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium artocarpi"              
[26] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Weeksellaceae;g__Chryseobacterium;s__Chryseobacterium sp024158915"            
[27] "d__Bacteria;p__Bacteroidota;c__Bacteroidia;o__Flavobacteriales;f__Wee